# RAG 基础管道 - 从零构建 PDF 问答系统

> **学习目标：**
> 1. 理解 RAG 完整流程：加载 → 分块 → Embedding → 存储 → 检索 → 生成
> 2. 掌握 ChromaDB 向量数据库操作
> 3. 使用 LangChain 构建 RAG 链
> 4. 实现文档问答系统
>
> **支持的后端：**
> - **Ollama（本地）**: `LLM_BACKEND=ollama`
> - **Groq（云端）**: `LLM_BACKEND=groq`
> - 通过 `.env` 文件切换，代码无需修改
>
> **前置条件：**
> - `conda activate llm-learn`
> - Embedding 始终使用本地 Ollama: `ollama pull nomic-embed-text`
> - 如果用 Ollama 后端: `ollama pull qwen3:8b`
> - 如果用 Groq 后端: 在 `.env` 中配置 `GROQ_API_KEY`
> - 选择 kernel 为 `llm-learn`

## 0. 导入依赖

In [ ]:
import sys
import os
import shutil

# 让 import config 和 rag 模块能找到项目根目录
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))

from config import chat, get_llm_config, get_embeddings, print_config
from rag_chain import get_langchain_llm, get_retriever, create_rag_chain, ask, rag_prompt
from document_loader import load_document, split_documents, create_vector_store, ingest_file, test_retrieval

from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
import chromadb
import numpy as np

# 显示当前配置
print_config()

---
## 1. 文档加载

RAG 的第一步：把文档读取为文本。LangChain 提供了多种 Loader：
- `TextLoader` - 纯文本文件
- `PyPDFLoader` - PDF 文件（按页拆分）
- `UnstructuredMarkdownLoader` - Markdown 文件

每个 Loader 返回 `Document` 对象列表，包含：
- `page_content`: 文本内容
- `metadata`: 元数据（来源、页码等）

In [ ]:
# 使用测试文档（纯文本）
loader = TextLoader("../data/sample_rag_test.txt", encoding="utf-8")
documents = loader.load()

print(f"加载了 {len(documents)} 个文档")
print(f"文档长度: {len(documents[0].page_content)} 字符")
print(f"元数据: {documents[0].metadata}")
print(f"\n前 200 字符预览:\n{documents[0].page_content[:200]}")

### 练习：加载 PDF（可选）

如果你有 PDF 文件，可以取消注释下面的代码试试：

In [ ]:
# # 加载 PDF 示例
# pdf_loader = PyPDFLoader("../data/your_document.pdf")
# pdf_docs = pdf_loader.load()
# print(f"PDF 共 {len(pdf_docs)} 页")
# for i, doc in enumerate(pdf_docs[:3]):
#     print(f"\n--- 第 {i+1} 页 (前100字) ---")
#     print(doc.page_content[:100])

---
## 2. 文本分块

为什么要分块？
- Embedding 模型有输入长度限制
- 小块文本更容易精确匹配用户的问题
- 大块文本包含太多无关内容，降低检索精度

**关键参数：**
- `chunk_size`: 每个块的最大字符数（建议 500-1000）
- `chunk_overlap`: 相邻块的重叠字符数（建议 50-100，防止信息截断）
- `separators`: 切分优先级，优先在段落/句子边界切分

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # 每块最大 500 字符
    chunk_overlap=50,     # 相邻块重叠 50 字符
    length_function=len,
    separators=["\n\n", "\n", "。", ".", " ", ""],  # 中文友好
)

chunks = text_splitter.split_documents(documents)

print(f"原始文档: {len(documents)} 个")
print(f"分块后: {len(chunks)} 个文本块")
print(f"\n--- 各块长度 ---")
for i, chunk in enumerate(chunks):
    print(f"块 {i}: {len(chunk.page_content)} 字符")

In [ ]:
# 查看前3个块的内容
for i, chunk in enumerate(chunks[:3]):
    print(f"\n{'='*50}")
    print(f"块 {i} ({len(chunk.page_content)} 字符)")
    print(f"{'='*50}")
    print(chunk.page_content)

### 实验：不同 chunk_size 的效果

试试不同的分块参数，观察对分块结果的影响：

In [ ]:
for size in [200, 500, 1000]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=50)
    result = splitter.split_documents(documents)
    avg_len = sum(len(c.page_content) for c in result) / len(result)
    print(f"chunk_size={size:>5} -> {len(result):>3} 块, 平均长度 {avg_len:.0f} 字符")

---
## 3. Embedding 向量化

Embedding 是 RAG 的核心：将文本转换为向量，使得语义相似的文本在向量空间中距离更近。

根据后端自动选择 Embedding 模型：
- **Ollama 后端**: `nomic-embed-text`（768 维，通过 Ollama 本地运行）
- **Groq 后端**: `all-MiniLM-L6-v2`（384 维，sentence-transformers，本地 CPU 运行，无需 Ollama）

In [ ]:
# 初始化 Embedding 模型（根据后端自动选择）
embeddings = get_embeddings()

# 测试：对单个文本生成 Embedding
test_text = "RAG 是一种将检索与生成结合的技术"
vector = embeddings.embed_query(test_text)

print(f"输入文本: {test_text}")
print(f"向量维度: {len(vector)}")
print(f"前10个值: {vector[:10]}")

In [ ]:
# 实验：语义相似度
# 看看语义相近的文本，向量距离是否更近

texts = [
    "RAG 是检索增强生成技术",        # 原始
    "Retrieval-Augmented Generation", # 英文同义
    "向量数据库用于存储 Embedding",   # 相关但不同
    "今天天气真好",                   # 完全无关
]

vectors = embeddings.embed_documents(texts)

# 计算与第一个文本的余弦相似度
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("与 '" + texts[0] + "' 的相似度：")
for i, text in enumerate(texts):
    sim = cosine_similarity(vectors[0], vectors[i])
    print(f"  {sim:.4f} | {text}")

---
## 4. 存入 ChromaDB 向量数据库

ChromaDB 是一个轻量级的向量数据库：
- 本地运行，无需部署
- 支持持久化到磁盘
- 与 LangChain 深度集成

两种使用方式：
1. **直接用 chromadb 客户端**（更底层，更灵活）
2. **用 LangChain 的 Chroma 封装**（更方便，适合 RAG 管道）

我们两种都试试：

In [ ]:
# 方式一：LangChain Chroma 封装（推荐，与 RAG 链无缝集成）
CHROMA_DB_PATH = "../chroma_db_notebook"

# 如果已存在，清理旧数据
import shutil
if os.path.exists(CHROMA_DB_PATH):
    shutil.rmtree(CHROMA_DB_PATH)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="rag_tutorial",
    persist_directory=CHROMA_DB_PATH,
)

print(f"已存入 {len(chunks)} 个文本块到 ChromaDB")
print(f"持久化路径: {CHROMA_DB_PATH}")

In [ ]:
# 方式二：直接用 chromadb 客户端（了解底层原理）
client = chromadb.PersistentClient(path=CHROMA_DB_PATH)
collection = client.get_collection("rag_tutorial")

print(f"Collection 名称: {collection.name}")
print(f"文档总数: {collection.count()}")

# 查看一条记录
sample = collection.peek(limit=1)
print(f"\n示例 ID: {sample['ids'][0]}")
print(f"示例文本: {sample['documents'][0][:100]}...")

---
## 5. 相似度检索

向量存好了，现在来检索！核心步骤：
1. 把用户问题转换为向量
2. 在 ChromaDB 中找到最相似的文本块
3. 返回 Top-K 个结果

In [ ]:
# 使用 LangChain 的 similarity_search
query = "RAG 的核心流程是什么？"

results = vectorstore.similarity_search_with_score(query, k=3)

print(f"查询: {query}\n")
for i, (doc, score) in enumerate(results):
    print(f"--- 结果 {i+1} (距离: {score:.4f}) ---")
    print(doc.page_content[:200])
    print()

In [ ]:
# 试试不同的问题
queries = [
    "什么是 Embedding？",
    "ChromaDB 和 Pinecone 有什么区别？",
    "chunk_size 应该设置多大？",
]

for q in queries:
    results = vectorstore.similarity_search(q, k=1)
    print(f"Q: {q}")
    print(f"A: {results[0].page_content[:150]}...")
    print()

---
## 6. 构建 RAG Chain（核心！）

把所有组件串起来：**检索 → 组装 Prompt → LLM 生成**

LangChain 的 `RetrievalQA` 自动完成这个流程：
1. 用户提问
2. Retriever 检索相关文档
3. 将文档 + 问题填入 Prompt 模板
4. 发送给 Qwen3 生成回答

In [ ]:
# Step 1: 创建 Retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},  # 返回 top-3 个结果
)

# Step 2: 定义 Prompt 模板（复用 rag_chain.py 中的模板）
prompt_template = rag_prompt
print(f"Prompt 模板:\n{prompt_template.template[:200]}...")

# Step 3: 初始化 LLM（自动根据 .env 选择 Ollama 或 Groq）
llm = get_langchain_llm(temperature=0.1, num_ctx=8192)

# Step 4: 组装 RAG Chain
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",  # 将所有检索到的文档拼接到 prompt
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt_template},
)

print("\nRAG Chain 构建完成！")

In [ ]:
# 测试 RAG 问答
question = "RAG 解决了大语言模型的什么问题？"

result = rag_chain.invoke({"query": question})

print(f"问题: {question}")
print(f"\n答案:\n{result['result']}")
print(f"\n--- 参考来源 ({len(result['source_documents'])} 个) ---")
for i, doc in enumerate(result["source_documents"]):
    print(f"[{i+1}] {doc.page_content[:100]}...")

In [ ]:
# 多个问题测试
questions = [
    "文本分块的关键参数有哪些？",
    "有哪些常见的向量数据库？",
    "RAG 中 Prompt 设计需要注意什么？",
    "如何优化 RAG 的检索质量？",
]

for q in questions:
    result = rag_chain.invoke({"query": q})
    print(f"\nQ: {q}")
    print(f"A: {result['result'][:300]}")
    print("-" * 60)

---
## 7. 对比实验：有 RAG vs 无 RAG

对比同一个问题，使用 RAG 和不使用 RAG 时模型回答的区别：

In [ ]:
question = "ChromaDB 支持哪些距离度量方式？"
cfg = get_llm_config()

# 无 RAG：直接问模型
print("=" * 50)
print(f"[无 RAG] 直接问 {cfg['backend']}:{cfg['model']}")
print("=" * 50)
response = chat([{"role": "user", "content": question}], temperature=0.1)
print(response)

# 有 RAG：先检索再回答
print(f"\n{'='*50}")
print("[有 RAG] 检索后回答")
print("=" * 50)
result = rag_chain.invoke({"query": question})
print(result["result"])

---
## 8. 使用已有模块（复用 Python 代码）

之前创建的 `document_loader.py` 和 `rag_chain.py` 可以直接 import 使用：

In [ ]:
# 使用模块化的方式运行完整 RAG 流程
# document_loader.py 和 rag_chain.py 已在 cell-0 导入

# Step 1: 文档入库（加载 → 分块 → 向量化）
docs = load_document("../data/sample_rag_test.txt")
chunks = split_documents(docs, chunk_size=500, chunk_overlap=50)
create_vector_store(chunks, collection_name="module_test", persist_directory="../chroma_db")

# Step 2: 构建 RAG 链（自动选择 Ollama 或 Groq 后端）
retriever = get_retriever(persist_directory="../chroma_db", collection_name="module_test")
chain = create_rag_chain(retriever)

# Step 3: 问答
ask(chain, "Embedding 模型有哪些选择？")

---
## 9. 总结与思考

### 本节学到了：
1. **文档加载**：使用 LangChain Loader 读取不同格式的文件
2. **文本分块**：`RecursiveCharacterTextSplitter` 按语义边界切分
3. **Embedding**：`nomic-embed-text` 将文本转为 768 维向量（始终本地 Ollama）
4. **向量存储**：ChromaDB 存储和检索向量
5. **RAG Chain**：`RetrievalQA` 串联检索和生成
6. **多后端支持**：通过 `config.py` 统一接口，Ollama 和 Groq 无缝切换

### 架构说明：
```
.env (LLM_BACKEND=ollama|groq)
  ↓
config.py → get_llm_config() → 统一配置
  ↓
rag_chain.py → get_langchain_llm() → 返回 OllamaLLM 或 ChatGroq
  ↓
document_loader.py → 文档加载/分块/向量化（Embedding 始终用本地 Ollama）
```

### 关键参数回顾：
| 参数 | 建议值 | 影响 | Ollama | Groq |
|------|--------|------|--------|------|
| chunk_size | 500-1000 | 分块大小，影响检索精度 | ✅ | ✅ |
| chunk_overlap | 50-100 | 块重叠，防止信息截断 | ✅ | ✅ |
| top_k | 3-5 | 检索结果数量 | ✅ | ✅ |
| temperature | 0.1-0.3 | RAG 用低温度减少幻觉 | ✅ | ✅ |
| num_ctx | 8192+ | 上下文窗口大小 | ✅ | ❌ 模型固定 |

### 下一步（03_rag_advanced）：
- Multi-Query Retrieval（多查询检索）
- Reranking（重排序）
- 对话式 RAG（保留历史上下文）

---
## 清理（可选）

In [ ]:
# 清理 notebook 产生的临时数据库
# import shutil
# shutil.rmtree("../chroma_db_notebook", ignore_errors=True)
# print("已清理临时数据")